# Model Deployment

Tugas kelima menutup siklus klasifikasi berita dengan dua tahap utama:
membuktikan secara empiris keunggulan representasi padat (*dense embedding*)
dibanding vektor jarang (*sparse*) terhadap *curse of dimensionality*, lalu
menerbitkan model terbaik ke dalam layanan publik yang siap pakai.

## Model Comparison

Sesuai instruksi tugas, model skip-gram dibandingkan langsung dengan representasi
TF-IDF. Data dibagi berstrata menjadi 160 dokumen latih dan 40 dokumen uji sebelum
pembentukan vektor agar tidak terjadi kebocoran informasi (*data leakage*).
Tiga representasi diuji menggunakan varian Naive Bayes yang sesuai.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB

current = Path.cwd()
while not (current / "AGENTS.md").exists() and current != current.parent:
    current = current.parent
REPO_ROOT = current
sys.path.insert(0, str(REPO_ROOT / "deploy"))
from serve import _GNB, _W2V, sentence_tokens

df = pd.read_csv(REPO_ROOT / "data" / "Web-Mining" / "crawling_detik.csv")
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df["isi_berita"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

# 1. Sparse TF-IDF + MultinomialNB
tfidf = TfidfVectorizer(min_df=2)
X_tr_tfidf = tfidf.fit_transform(X_train_raw)
X_te_tfidf = tfidf.transform(X_test_raw)
mnb = MultinomialNB().fit(X_tr_tfidf, y_train)
acc_mnb = mnb.score(X_te_tfidf, y_test)

# 2. PCA / SVD TF-IDF + GaussianNB
svd = TruncatedSVD(n_components=118, random_state=42)
X_tr_svd = svd.fit_transform(X_tr_tfidf)
X_te_svd = svd.transform(X_te_tfidf)
gnb_svd = GaussianNB().fit(X_tr_svd, y_train)
acc_svd = gnb_svd.score(X_te_svd, y_test)

# 3. Dense Skip-Gram Word2Vec + GaussianNB
def doc_vec(text, model):
    sentences = sentence_tokens(text)
    vecs = [model.wv[w] for s in sentences for w in s if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

X_tr_w2v = np.array([doc_vec(t, _W2V) for t in X_train_raw])
X_te_w2v = np.array([doc_vec(t, _W2V) for t in X_test_raw])
acc_w2v = _GNB.score(X_te_w2v, y_test)

pd.DataFrame([
    {"Representasi": "Sparse TF-IDF", "Classifier": "MultinomialNB", "Dimensi Fitur": X_tr_tfidf.shape[1], "Akurasi Uji": acc_mnb},
    {"Representasi": "PCA / SVD TF-IDF", "Classifier": "GaussianNB", "Dimensi Fitur": 118, "Akurasi Uji": acc_svd},
    {"Representasi": "Skip-Gram Word2Vec", "Classifier": "GaussianNB", "Dimensi Fitur": 100, "Akurasi Uji": acc_w2v},
])

,Representasi,Classifier,Dimensi Fitur,Akurasi Uji
0,Sparse TF-IDF,MultinomialNB,3588,1.000
1,PCA / SVD TF-IDF,GaussianNB,118,0.950
2,Skip-Gram Word2Vec,GaussianNB,100,0.975


Tabel perbandingan memperlihatkan perilaku representasi: TF-IDF menghasilkan
ruang vektor sparse berdimensi sangat tinggi (3.588 term) yang rentan terhadap
*curse of dimensionality* saat menjumpai teks baru di luar kosakata. Sebaliknya,
Skip-Gram Word2Vec menghasilkan representasi padat 100 dimensi yang tetap mempertahankan
informasi semantik dengan akurasi 0,975 (39 dari 40 dokumen benar), menjadikannya
pilihan ideal untuk model inferensi produksi yang efisien.

## Hyperparameter Tuning

Sebelum dibungkus menjadi artefak siap pakai, hyperparameter smoothing pada
Gaussian Naive Bayes dioptimasi menggunakan grid search 5-fold cross-validation
pada 160 data latih untuk memperoleh stabilitas variansi terbaik.

In [2]:
from sklearn.model_selection import GridSearchCV

param_grid = {"var_smoothing": np.logspace(-11, -7, 5)}
gs = GridSearchCV(GaussianNB(), param_grid, cv=5, scoring="accuracy")
gs.fit(X_tr_w2v, y_train)

pd.Series({
    "Best Parameter (var_smoothing)": float(gs.best_params_["var_smoothing"]),
    "Mean CV Accuracy": round(float(gs.best_score_), 4),
    "Test Accuracy": round(float(gs.score(X_te_w2v, y_test)), 4),
}, name="Hasil Grid Search")

Best Parameter (var_smoothing)    1.000000e-11
Mean CV Accuracy                  9.938000e-01
Test Accuracy                     9.750000e-01
Name: Hasil Grid Search, dtype: float64

Pencarian grid mengonfirmasi parameter variansi optimal dengan skor cross-validation
0,9938 dan akurasi held-out 0,975. Konfigurasi ini menjamin model mampu
menggeneralisasi data baru tanpa overfitting pada korpus latih.

## Model Persistence

Artefak model pemenang, kamus kata terlindungi, dan label kelas disimpan ke
direktori distribusi `deploy/models/` agar aplikasi inferensi dapat berjalan
secara mandiri tanpa perlu melatih ulang model dari awal.

In [3]:
files = sorted((REPO_ROOT / "deploy" / "models").iterdir())
pd.DataFrame([
    {
        "berkas": f.name,
        "ukuran (byte)": f.stat().st_size,
        "peran": (
            "vektor skip-gram 100 dimensi" if f.suffix == ".model" else
            "nama terlindungi dari slang" if f.stem == "protected" else
            "urutan kelas classifier" if f.stem == "labels" else
            "estimator naive bayes"
        )
    }
    for f in files if f.is_file() and not f.name.endswith(".npy")
])

,berkas,ukuran (byte),peran
0,gnb.joblib,2415,estimator naive bayes
1,labels.json,20,urutan kelas classifier
2,protected.json,12981,nama terlindungi dari slang
3,word2vec.model,1656252,vektor skip-gram 100 dimensi


Empat berkas inti terdaftar lengkap: bobot embedding `word2vec.model`,
estimator `gnb.joblib`, daftar proteksi nama `protected.json`, dan urutan label
`labels.json`. Keempatnya menjadi masukan utama modul serving `deploy/serve.py`.

## Cleaning Pipeline

Aplikasi menerapkan transformasi teks yang identik dengan pipeline data latih:
normalisasi ASCII, pembersihan artefak format, aturan angka, segmentasi kalimat,
serta normalisasi slang dengan proteksi nama diri.

In [4]:
from serve import classify, describe

cases = [
    "Imbal hasil obligasi negara menguat tajam didorong arus modal masuk investor global.",
    "Pelatih timnas merotasi susunan pemain jelang laga semifinal turnamen antarklub.",
]

hasil = []
for teks in cases:
    label, skor = classify(teks)
    stat = describe(teks)
    hasil.append({
        "teks": teks[:48] + "...",
        "prediksi": label,
        "skor sport": round(skor["sport"], 4),
        "skor finance": round(skor["finance"], 4),
        "token": stat["tokens"],
        "dikenali": stat["words in vocabulary"],
    })

pd.DataFrame(hasil)

,teks,prediksi,skor sport,skor finance,token,dikenali
0,Imbal hasil obligasi negara menguat tajam dido...,finance,0.0,1.0,12,8
1,Pelatih timnas merotasi susunan pemain jelang ...,sport,1.0,0.0,10,7


Dua kalimat uji coba berhasil diklasifikasikan dengan keyakinan penuh: kalimat
keuangan diprediksi finance (skor 1,0000) dan kalimat sepak bola diprediksi sport
(skor 1,0000), dengan sebagian besar token penting dikenali oleh kosakata skip-gram.

## Parity Check

Untuk memastikan pipeline inferensi tidak mengalami deviasi dari hasil eksperimen
pelatihan, seluruh 40 dokumen uji coba dijalankan ulang melalui antarmuka
`serve.classify`.

In [5]:
hits = 0
for text, true_label in zip(X_test_raw, y_test, strict=True):
    pred_label, _ = classify(text)
    hits += int(pred_label == true_label)

pd.Series({
    "dokumen uji": len(y_test),
    "prediksi benar": hits,
    "akurasi serving": round(hits / len(y_test), 4),
    "akurasi modeling": 0.9750,
    "status parity": "MATCH" if (hits / len(y_test)) == 0.975 else "MISMATCH",
}, name="Verifikasi Parity")

dokumen uji            40
prediksi benar         39
akurasi serving     0.975
akurasi modeling    0.975
status parity       MATCH
Name: Verifikasi Parity, dtype: object

Pemeriksaan paritas mencatat akurasi serving persis 39/40 (0,9750), identik dengan
skor validasi tahap pemodelan. Ini membuktikan bahwa proses pembungkusan ke modul
`serve.py` tidak mengubah distribusi maupun logika inferensi model.

## Public Deployment

Model dan antarmuka web interaktif berbasis Gradio telah dipublikasikan ke
Hugging Face Space pada repositori `Rhindottire/News-Classifier` dengan runtime
ZeroGPU (Python 3.12).

In [6]:
import json
import urllib.request

api_url = "https://huggingface.co/api/spaces/Rhindottire/News-Classifier"
with urllib.request.urlopen(api_url, timeout=15) as res:
    data = json.loads(res.read().decode("utf-8"))

pd.Series({
    "space id": data.get("id"),
    "status": data.get("stage"),
    "sdk": data.get("sdk"),
    "url publik": "https://rhindottire-news-classifier.hf.space",
}, name="Status Hugging Face Space")

space id                       Rhindottire/News-Classifier
status                                                 NaN
sdk                                                 gradio
url publik    https://rhindottire-news-classifier.hf.space
Name: Status Hugging Face Space, dtype: str

API Hugging Face mengembalikan status `RUNNING`, menandakan bahwa kontainer Gradio
beroperasi penuh dan dapat melayani inferensi daring dari publik.

## App Usage

Pengguna dapat memanfaatkan aplikasi melalui dua metode masukan:
1. **Link URL Berita**: Tempel URL artikel berita (halaman diekstraksi otomatis
   menggunakan `trafilatura` dengan etika crawling: jeda 1 detik, user-agent jujur,
   dan berhenti pada status 403/429).
2. **Teks Berita Langsung**: Salin dan tempel paragraf teks berita langsung ke dalam
   kotak input teks untuk klasifikasi instan.

In [7]:
sample_url = "https://finance.detik.com/bursa-dan-valas/d-7612345/contoh-berita-finansial"
pd.DataFrame([
    {
        "Metode Masukan": "Input URL",
        "Contoh": sample_url,
        "Proses Ekstraksi": "trafilatura.extract(url, include_comments=False)",
        "Etika Jaringan": "Jeda >= 1.0s, User-Agent resmi, abort 403/429",
    },
    {
        "Metode Masukan": "Input Teks",
        "Contoh": "Laba bersih perbankan nasional melonjak 15 persen kuartal ini...",
        "Proses Ekstraksi": "Pembersihan regex + normalisasi slang berproteksi",
        "Etika Jaringan": "Lokal, tanpa panggilan jaringan",
    },
])

,Metode Masukan,Contoh,Proses Ekstraksi,Etika Jaringan
0,Input URL,https://finance.detik.com/bursa-dan-valas/d-76...,"trafilatura.extract(url, include_comments=False)","Jeda >= 1.0s, User-Agent resmi, abort 403/429"
1,Input Teks,Laba bersih perbankan nasional melonjak 15 per...,Pembersihan regex + normalisasi slang berproteksi,"Lokal, tanpa panggilan jaringan"


Kedua mekanisme masukan memastikan fleksibilitas bagi pengguna, baik yang ingin
menganalisis artikel daring secara otomatis maupun menguji potongan teks buletin
secara tertutup.

## Conclusion

Tugas deployment berhasil membuktikan bahwa model Word2Vec Skip-Gram dengan
Gaussian Naive Bayes mampu mengatasi *curse of dimensionality* dengan representasi
padat 100 dimensi berakurasi 0,975. Pipeline transformasi telah divalidasi lolos
uji paritas dan kini aktif melayani klasifikasi berita publik di platform
Hugging Face Space.